In [ ]:
import os
import sys
import json
import glob
import math
import shutil
import numpy as np
import pandas as pd
import cv2
from scipy import signal
from scipy.signal import periodogram
from tqdm import tqdm
import torch
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

REPO_ROOT = "/home/iec/MinhHieu/Non-Invasive/rPPG"


## Inlined source

The cells below contain the model / loss source that was previously imported from the `neural_methods/` (and `evaluation/`) packages. They are inlined here so the notebook is self-contained.


In [ ]:
# === inlined from neural_methods/model/DeepPhys.py ===
"""DeepPhys - 2D Convolutional Attention Network.
DeepPhys: Video-Based Physiological Measurement Using Convolutional Attention Networks
ECCV, 2018
Weixuan Chen, Daniel McDuff
"""

import torch
import torch.nn as nn


class Attention_mask(nn.Module):
    def __init__(self):
        super(Attention_mask, self).__init__()

    def forward(self, x):
        xsum = torch.sum(x, dim=2, keepdim=True)
        xsum = torch.sum(xsum, dim=3, keepdim=True)
        xshape = tuple(x.size())
        return x / xsum * xshape[2] * xshape[3] * 0.5

    def get_config(self):
        """May be generated manually. """
        config = super(Attention_mask, self).get_config()
        return config


class DeepPhys(nn.Module):

    def __init__(self, in_channels=3, nb_filters1=32, nb_filters2=64, kernel_size=3, dropout_rate1=0.25,
                 dropout_rate2=0.5, pool_size=(2, 2), nb_dense=128, img_size=36):
        """Definition of DeepPhys.
        Args:
          in_channels: the number of input channel. Default: 3
          img_size: height/width of each frame. Default: 36.
        Returns:
          DeepPhys model.
        """
        super(DeepPhys, self).__init__()
        self.in_channels = in_channels
        self.kernel_size = kernel_size
        self.dropout_rate1 = dropout_rate1
        self.dropout_rate2 = dropout_rate2
        self.pool_size = pool_size
        self.nb_filters1 = nb_filters1
        self.nb_filters2 = nb_filters2
        self.nb_dense = nb_dense
        # Motion branch convs
        self.motion_conv1 = nn.Conv2d(self.in_channels, self.nb_filters1, kernel_size=self.kernel_size, padding=(1, 1),
                                      bias=True)
        self.motion_conv2 = nn.Conv2d(self.nb_filters1, self.nb_filters1, kernel_size=self.kernel_size, bias=True)
        self.motion_conv3 = nn.Conv2d(self.nb_filters1, self.nb_filters2, kernel_size=self.kernel_size, padding=(1, 1),
                                      bias=True)
        self.motion_conv4 = nn.Conv2d(self.nb_filters2, self.nb_filters2, kernel_size=self.kernel_size, bias=True)
        # Apperance branch convs
        self.apperance_conv1 = nn.Conv2d(self.in_channels, self.nb_filters1, kernel_size=self.kernel_size,
                                         padding=(1, 1), bias=True)
        self.apperance_conv2 = nn.Conv2d(self.nb_filters1, self.nb_filters1, kernel_size=self.kernel_size, bias=True)
        self.apperance_conv3 = nn.Conv2d(self.nb_filters1, self.nb_filters2, kernel_size=self.kernel_size,
                                         padding=(1, 1), bias=True)
        self.apperance_conv4 = nn.Conv2d(self.nb_filters2, self.nb_filters2, kernel_size=self.kernel_size, bias=True)
        # Attention layers
        self.apperance_att_conv1 = nn.Conv2d(self.nb_filters1, 1, kernel_size=1, padding=(0, 0), bias=True)
        self.attn_mask_1 = Attention_mask()
        self.apperance_att_conv2 = nn.Conv2d(self.nb_filters2, 1, kernel_size=1, padding=(0, 0), bias=True)
        self.attn_mask_2 = Attention_mask()
        # Avg pooling
        self.avg_pooling_1 = nn.AvgPool2d(self.pool_size)
        self.avg_pooling_2 = nn.AvgPool2d(self.pool_size)
        self.avg_pooling_3 = nn.AvgPool2d(self.pool_size)
        # Dropout layers
        self.dropout_1 = nn.Dropout(self.dropout_rate1)
        self.dropout_2 = nn.Dropout(self.dropout_rate1)
        self.dropout_3 = nn.Dropout(self.dropout_rate1)
        self.dropout_4 = nn.Dropout(self.dropout_rate2)
        # Dense layers
        if img_size == 36:
            self.final_dense_1 = nn.Linear(3136, self.nb_dense, bias=True)
        elif img_size == 72:
            self.final_dense_1 = nn.Linear(16384, self.nb_dense, bias=True)
        elif img_size == 96:
            self.final_dense_1 = nn.Linear(30976, self.nb_dense, bias=True)
        else:
            raise Exception('Unsupported image size')
        self.final_dense_2 = nn.Linear(self.nb_dense, 1, bias=True)

    def forward(self, inputs, params=None):

        diff_input = inputs[:, :3, :, :]
        raw_input = inputs[:, 3:, :, :]

        d1 = torch.tanh(self.motion_conv1(diff_input))
        d2 = torch.tanh(self.motion_conv2(d1))

        r1 = torch.tanh(self.apperance_conv1(raw_input))
        r2 = torch.tanh(self.apperance_conv2(r1))

        g1 = torch.sigmoid(self.apperance_att_conv1(r2))
        g1 = self.attn_mask_1(g1)
        gated1 = d2 * g1

        d3 = self.avg_pooling_1(gated1)
        d4 = self.dropout_1(d3)

        r3 = self.avg_pooling_2(r2)
        r4 = self.dropout_2(r3)

        d5 = torch.tanh(self.motion_conv3(d4))
        d6 = torch.tanh(self.motion_conv4(d5))

        r5 = torch.tanh(self.apperance_conv3(r4))
        r6 = torch.tanh(self.apperance_conv4(r5))

        g2 = torch.sigmoid(self.apperance_att_conv2(r6))
        g2 = self.attn_mask_2(g2)
        gated2 = d6 * g2

        d7 = self.avg_pooling_3(gated2)
        d8 = self.dropout_3(d7)
        d9 = d8.view(d8.size(0), -1)
        d10 = torch.tanh(self.final_dense_1(d9))
        d11 = self.dropout_4(d10)
        out = self.final_dense_2(d11)

        return out



In [ ]:
# === inlined from neural_methods/model/TS_CAN.py ===
"""Temporal Shift Convolutional Attention Network (TS-CAN).
Multi-Task Temporal Shift Attention Networks for On-Device Contactless Vitals Measurement
NeurIPS, 2020
Xin Liu, Josh Fromm, Shwetak Patel, Daniel McDuff
"""

import torch
import torch.nn as nn


class Attention_mask(nn.Module):
    def __init__(self):
        super(Attention_mask, self).__init__()

    def forward(self, x):
        xsum = torch.sum(x, dim=2, keepdim=True)
        xsum = torch.sum(xsum, dim=3, keepdim=True)
        xshape = tuple(x.size())
        return x / xsum * xshape[2] * xshape[3] * 0.5

    def get_config(self):
        """May be generated manually. """
        config = super(Attention_mask, self).get_config()
        return config


class TSM(nn.Module):
    def __init__(self, n_segment=10, fold_div=3):
        super(TSM, self).__init__()
        self.n_segment = n_segment
        self.fold_div = fold_div

    def forward(self, x):
        nt, c, h, w = x.size()
        n_batch = nt // self.n_segment
        x = x.view(n_batch, self.n_segment, c, h, w)
        fold = c // self.fold_div
        out = torch.zeros_like(x)
        out[:, :-1, :fold] = x[:, 1:, :fold]  # shift left
        out[:, 1:, fold: 2 * fold] = x[:, :-1, fold: 2 * fold]  # shift right
        out[:, :, 2 * fold:] = x[:, :, 2 * fold:]  # not shift
        return out.view(nt, c, h, w)


class TSCAN(nn.Module):

    def __init__(self, in_channels=3, nb_filters1=32, nb_filters2=64, kernel_size=3, dropout_rate1=0.25,
                 dropout_rate2=0.5, pool_size=(2, 2), nb_dense=128, frame_depth=20, img_size=36):
        """Definition of TS_CAN.
        Args:
          in_channels: the number of input channel. Default: 3
          frame_depth: the number of frame (window size) used in temport shift. Default: 20
          img_size: height/width of each frame. Default: 36.
        Returns:
          TS_CAN model.
        """
        super(TSCAN, self).__init__()
        self.in_channels = in_channels
        self.kernel_size = kernel_size
        self.dropout_rate1 = dropout_rate1
        self.dropout_rate2 = dropout_rate2
        self.pool_size = pool_size
        self.nb_filters1 = nb_filters1
        self.nb_filters2 = nb_filters2
        self.nb_dense = nb_dense
        # TSM layers
        self.TSM_1 = TSM(n_segment=frame_depth)
        self.TSM_2 = TSM(n_segment=frame_depth)
        self.TSM_3 = TSM(n_segment=frame_depth)
        self.TSM_4 = TSM(n_segment=frame_depth)
        # Motion branch convs
        self.motion_conv1 = nn.Conv2d(self.in_channels, self.nb_filters1, kernel_size=self.kernel_size, padding=(1, 1),
                                      bias=True)
        self.motion_conv2 = nn.Conv2d(
            self.nb_filters1, self.nb_filters1, kernel_size=self.kernel_size, bias=True)
        self.motion_conv3 = nn.Conv2d(self.nb_filters1, self.nb_filters2, kernel_size=self.kernel_size, padding=(1, 1),
                                      bias=True)
        self.motion_conv4 = nn.Conv2d(
            self.nb_filters2, self.nb_filters2, kernel_size=self.kernel_size, bias=True)
        # Apperance branch convs
        self.apperance_conv1 = nn.Conv2d(self.in_channels, self.nb_filters1, kernel_size=self.kernel_size,
                                         padding=(1, 1), bias=True)
        self.apperance_conv2 = nn.Conv2d(
            self.nb_filters1, self.nb_filters1, kernel_size=self.kernel_size, bias=True)
        self.apperance_conv3 = nn.Conv2d(self.nb_filters1, self.nb_filters2, kernel_size=self.kernel_size,
                                         padding=(1, 1), bias=True)
        self.apperance_conv4 = nn.Conv2d(
            self.nb_filters2, self.nb_filters2, kernel_size=self.kernel_size, bias=True)
        # Attention layers
        self.apperance_att_conv1 = nn.Conv2d(
            self.nb_filters1, 1, kernel_size=1, padding=(0, 0), bias=True)
        self.attn_mask_1 = Attention_mask()
        self.apperance_att_conv2 = nn.Conv2d(
            self.nb_filters2, 1, kernel_size=1, padding=(0, 0), bias=True)
        self.attn_mask_2 = Attention_mask()
        # Avg pooling
        self.avg_pooling_1 = nn.AvgPool2d(self.pool_size)
        self.avg_pooling_2 = nn.AvgPool2d(self.pool_size)
        self.avg_pooling_3 = nn.AvgPool2d(self.pool_size)
        # Dropout layers
        self.dropout_1 = nn.Dropout(self.dropout_rate1)
        self.dropout_2 = nn.Dropout(self.dropout_rate1)
        self.dropout_3 = nn.Dropout(self.dropout_rate1)
        self.dropout_4 = nn.Dropout(self.dropout_rate2)
        # Dense layers
        if img_size == 36:
            self.final_dense_1 = nn.Linear(3136, self.nb_dense, bias=True)
        elif img_size == 72:
            self.final_dense_1 = nn.Linear(16384, self.nb_dense, bias=True)
        elif img_size == 96:
            self.final_dense_1 = nn.Linear(30976, self.nb_dense, bias=True)
        elif img_size == 128:
            self.final_dense_1 = nn.Linear(57600, self.nb_dense, bias=True)
        else:
            raise Exception('Unsupported image size')
        self.final_dense_2 = nn.Linear(self.nb_dense, 1, bias=True)

    def forward(self, inputs, params=None):
        diff_input = inputs[:, :3, :, :]
        raw_input = inputs[:, 3:, :, :]

        diff_input = self.TSM_1(diff_input)
        d1 = torch.tanh(self.motion_conv1(diff_input))
        d1 = self.TSM_2(d1)
        d2 = torch.tanh(self.motion_conv2(d1))

        r1 = torch.tanh(self.apperance_conv1(raw_input))
        r2 = torch.tanh(self.apperance_conv2(r1))

        g1 = torch.sigmoid(self.apperance_att_conv1(r2))
        g1 = self.attn_mask_1(g1)
        gated1 = d2 * g1

        d3 = self.avg_pooling_1(gated1)
        d4 = self.dropout_1(d3)

        r3 = self.avg_pooling_2(r2)
        r4 = self.dropout_2(r3)

        d4 = self.TSM_3(d4)
        d5 = torch.tanh(self.motion_conv3(d4))
        d5 = self.TSM_4(d5)
        d6 = torch.tanh(self.motion_conv4(d5))

        r5 = torch.tanh(self.apperance_conv3(r4))
        r6 = torch.tanh(self.apperance_conv4(r5))

        g2 = torch.sigmoid(self.apperance_att_conv2(r6))
        g2 = self.attn_mask_2(g2)
        gated2 = d6 * g2

        d7 = self.avg_pooling_3(gated2)
        d8 = self.dropout_3(d7)
        d9 = d8.view(d8.size(0), -1)
        d10 = torch.tanh(self.final_dense_1(d9))
        d11 = self.dropout_4(d10)
        out = self.final_dense_2(d11)

        return out


class MTTS_CAN(nn.Module):
    """MTTS_CAN is the multi-task (respiration) version of TS-CAN"""

    def __init__(self, in_channels=3, nb_filters1=32, nb_filters2=64, kernel_size=3, dropout_rate1=0.25,
                 dropout_rate2=0.5, pool_size=(2, 2), nb_dense=128, frame_depth=20):
        super(MTTS_CAN, self).__init__()
        self.in_channels = in_channels
        self.kernel_size = kernel_size
        self.dropout_rate1 = dropout_rate1
        self.dropout_rate2 = dropout_rate2
        self.pool_size = pool_size
        self.nb_filters1 = nb_filters1
        self.nb_filters2 = nb_filters2
        self.nb_dense = nb_dense
        # TSM layers
        self.TSM_1 = TSM(n_segment=frame_depth)
        self.TSM_2 = TSM(n_segment=frame_depth)
        self.TSM_3 = TSM(n_segment=frame_depth)
        self.TSM_4 = TSM(n_segment=frame_depth)
        # Motion branch convs
        self.motion_conv1 = nn.Conv2d(self.in_channels, self.nb_filters1, kernel_size=self.kernel_size, padding=(1, 1),
                                      bias=True)
        self.motion_conv2 = nn.Conv2d(
            self.nb_filters1, self.nb_filters1, kernel_size=self.kernel_size, bias=True)
        self.motion_conv3 = nn.Conv2d(self.nb_filters1, self.nb_filters2, kernel_size=self.kernel_size, padding=(1, 1),
                                      bias=True)
        self.motion_conv4 = nn.Conv2d(
            self.nb_filters2, self.nb_filters2, kernel_size=self.kernel_size, bias=True)
        # Apperance branch convs
        self.apperance_conv1 = nn.Conv2d(self.in_channels, self.nb_filters1, kernel_size=self.kernel_size,
                                         padding=(1, 1), bias=True)
        self.apperance_conv2 = nn.Conv2d(
            self.nb_filters1, self.nb_filters1, kernel_size=self.kernel_size, bias=True)
        self.apperance_conv3 = nn.Conv2d(self.nb_filters1, self.nb_filters2, kernel_size=self.kernel_size,
                                         padding=(1, 1), bias=True)
        self.apperance_conv4 = nn.Conv2d(
            self.nb_filters2, self.nb_filters2, kernel_size=self.kernel_size, bias=True)
        # Attention layers
        self.apperance_att_conv1 = nn.Conv2d(
            self.nb_filters1, 1, kernel_size=1, padding=(0, 0), bias=True)
        self.attn_mask_1 = Attention_mask()
        self.apperance_att_conv2 = nn.Conv2d(
            self.nb_filters2, 1, kernel_size=1, padding=(0, 0), bias=True)
        self.attn_mask_2 = Attention_mask()
        # Avg pooling
        self.avg_pooling_1 = nn.AvgPool2d(self.pool_size)
        self.avg_pooling_2 = nn.AvgPool2d(self.pool_size)
        self.avg_pooling_3 = nn.AvgPool2d(self.pool_size)
        # Dropout layers
        self.dropout_1 = nn.Dropout(self.dropout_rate1)
        self.dropout_2 = nn.Dropout(self.dropout_rate1)
        self.dropout_3 = nn.Dropout(self.dropout_rate1)
        self.dropout_4_y = nn.Dropout(self.dropout_rate2)
        self.dropout_4_r = nn.Dropout(self.dropout_rate2)

        # Dense layers
        self.final_dense_1_y = nn.Linear(16384, self.nb_dense, bias=True)
        self.final_dense_2_y = nn.Linear(self.nb_dense, 1, bias=True)
        self.final_dense_1_r = nn.Linear(16384, self.nb_dense, bias=True)
        self.final_dense_2_r = nn.Linear(self.nb_dense, 1, bias=True)

    def forward(self, inputs, params=None):
        diff_input = inputs[:, :3, :, :]
        raw_input = inputs[:, 3:, :, :]

        diff_input = self.TSM_1(diff_input)
        d1 = torch.tanh(self.motion_conv1(diff_input))
        d1 = self.TSM_2(d1)
        d2 = torch.tanh(self.motion_conv2(d1))

        r1 = torch.tanh(self.apperance_conv1(raw_input))
        r2 = torch.tanh(self.apperance_conv2(r1))

        g1 = torch.sigmoid(self.apperance_att_conv1(r2))
        g1 = self.attn_mask_1(g1)
        gated1 = d2 * g1

        d3 = self.avg_pooling_1(gated1)
        d4 = self.dropout_1(d3)

        r3 = self.avg_pooling_2(r2)
        r4 = self.dropout_2(r3)

        d4 = self.TSM_3(d4)
        d5 = torch.tanh(self.motion_conv3(d4))
        d5 = self.TSM_4(d5)
        d6 = torch.tanh(self.motion_conv4(d5))

        r5 = torch.tanh(self.apperance_conv3(r4))
        r6 = torch.tanh(self.apperance_conv4(r5))

        g2 = torch.sigmoid(self.apperance_att_conv2(r6))
        g2 = self.attn_mask_2(g2)
        gated2 = d6 * g2

        d7 = self.avg_pooling_3(gated2)
        d8 = self.dropout_3(d7)
        d9 = d8.view(d8.size(0), -1)

        d10 = torch.tanh(self.final_dense_1_y(d9))
        d11 = self.dropout_4_y(d10)
        out_y = self.final_dense_2_y(d11)

        d10 = torch.tanh(self.final_dense_1_r(d9))
        d11 = self.dropout_4_r(d10)
        out_r = self.final_dense_2_r(d11)

        return out_y, out_r


In [ ]:
# ----- paths -----
RAW_DATA_PATH       = os.path.join(REPO_ROOT, "data/Normal")
PREPROCESSED_PATH   = os.path.join(REPO_ROOT, "preprocessed_data/Normal/groupA")
OUTPUT_DIR          = os.path.join(REPO_ROOT, "results/Normal/groupA")

# ----- video / signal params -----
VIDEO_FPS    = 30       # camera frame rate
PPG_FS       = 60       # PPG sensor sampling rate (Hz)

# ----- Group A preprocessing params -----
CHUNK_LENGTH = 180      # frames per clip (6 s at 30 fps)
IMG_H, IMG_W = 72, 72   # face crop resolution
LABEL_TYPE   = "DiffNormalized"  # needs cumsum in post-processing
DATA_FORMAT  = "NDCHW"  # (N, D, C, H, W)
NUM_CHANNELS = 6        # DiffNormalized (3) + Standardized (3)

# ----- device -----
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

os.makedirs(PREPROCESSED_PATH, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("PREPROCESSED_PATH:", PREPROCESSED_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
# Model selection toggle
# Uncomment additional models as needed; each entry is (display_name, model_class_key, weight_path).
# model_class_key: "DeepPhys" or "Tscan"

MODELS = [
    ("PURE_DeepPhys",              "DeepPhys", "final_model_release/PURE_DeepPhys.pth"),
    ("PURE_TSCAN",                 "Tscan",    "final_model_release/PURE_TSCAN.pth"),
    ("SCAMPS_DeepPhys",          "DeepPhys", "final_model_release/SCAMPS_DeepPhys.pth"),
    ("SCAMPS_TSCAN",             "Tscan",    "final_model_release/SCAMPS_TSCAN.pth"),
    ("UBFC-rPPG_DeepPhys",       "DeepPhys", "final_model_release/UBFC-rPPG_DeepPhys.pth"),
    ("UBFC-rPPG_TSCAN",          "Tscan",    "final_model_release/UBFC-rPPG_TSCAN.pth"),
    ("BP4D_PseudoLabel_DeepPhys","DeepPhys", "final_model_release/BP4D_PseudoLabel_DeepPhys.pth"),
    ("BP4D_PseudoLabel_TSCAN",   "Tscan",    "final_model_release/BP4D_PseudoLabel_TSCAN.pth"),
    ("MA-UBFC_deepphys",         "DeepPhys", "final_model_release/MA-UBFC_deepphys.pth"),
    ("MA-UBFC_tscan",            "Tscan",    "final_model_release/MA-UBFC_tscan.pth"),
]

# TS-CAN frame_depth (TSM window size) — must match training config
TSCAN_FRAME_DEPTH = 10

print(f"Selected {len(MODELS)} model(s):")
for name, cls, path in MODELS:
    print(f"  {name}  ({cls})  ->  {path}")

In [ ]:
# Read video framesdef read_video_frames(video_path):    """Read all frames from a video file (MP4, MKV, AVI, etc.).    Returns:        frames (np.ndarray): shape (T, H, W, 3), dtype uint8, RGB order.    """    cap = cv2.VideoCapture(video_path)    if not cap.isOpened():        raise IOError(f"Cannot open video: {video_path}")    frames = []    while True:        ret, frame = cap.read()        if not ret:            break        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))    cap.release()    if not frames:        raise ValueError(f"Empty video: {video_path}")    return np.stack(frames, axis=0)

In [ ]:
def read_ppg_synced(session_path, num_frames):
    """
    Reads the PPG signal from 'ppg.csv' and resamples it to match the exact 
    timestamps of the video frames from 'frame_timestamps.csv'.
    """
    import pandas as pd
    import numpy as np
    import os
    
    # 1. Read the video frame timestamps
    frame_df = pd.read_csv(os.path.join(session_path, "frame_timestamps.csv"))
    
    # Validation
    if len(frame_df) != num_frames:
        print(f"Warning: Video has {num_frames} frames, but frame_timestamps.csv has {len(frame_df)} rows. Using min count.")
        min_len = min(len(frame_df), num_frames)
        frame_t = frame_df["timestamp"].values[:min_len]
    else:
        frame_t = frame_df["timestamp"].values

    # 2. Read the raw PPG data
    ppg_df = pd.read_csv(os.path.join(session_path, "ppg.csv"))
    
    ppg_t = ppg_df["Timestamp"].values
    ppg_val = ppg_df["PPG"].values
    
    # Clip frame times to valid ppg range to avoid extrapolation
    frame_t_clipped = np.clip(frame_t, ppg_t[0], ppg_t[-1])
    
    # 3. Resample (Interpolate)
    ppg_resampled = np.interp(frame_t_clipped, ppg_t, ppg_val)
    
    return ppg_resampled.astype(np.float32)

In [ ]:
# Normalization functions

def diff_normalize_data(data):
    """DiffNormalized: (frame[t+1]-frame[t]) / (frame[t+1]+frame[t]+1e-7), then / std."""
    data = data.astype(np.float32)
    n, h, w, c = data.shape
    out = np.zeros_like(data)
    out[:n - 1] = (data[1:] - data[:-1]) / (data[1:] + data[:-1] + 1e-7)
    std = np.std(out)
    if std > 0:
        out /= std
    return out


def standardized_data(data):
    """Standardized: global z-score over all pixels and frames."""
    data = data.astype(np.float32)
    m = np.mean(data)
    s = np.std(data)
    if s > 0:
        data = (data - m) / s
    else:
        data = np.zeros_like(data)
    data = np.where(np.isnan(data), np.zeros_like(data), data)
    return data


def diff_normalize_label(label):
    """DiffNormalized label: finite difference normalised by std, zero-padded."""
    diff = np.diff(label.astype(np.float64), axis=0)
    s = np.std(diff)
    if s > 0:
        diff = diff / s
    return np.append(diff, [0.0]).astype(np.float32)

In [ ]:
# Face crop + resize

def crop_face_resize(frames, out_h, out_w, large_box_coef=1.5):
    """Detect face on frame 0 with Haar Cascade, expand bbox by coef, resize all frames."""
    xml_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector  = cv2.CascadeClassifier(xml_path)

    frame0 = frames[0]
    if frame0.dtype != np.uint8:
        frame0 = np.clip(frame0, 0, 255).astype(np.uint8)
    gray = cv2.cvtColor(frame0, cv2.COLOR_RGB2GRAY)

    faces = detector.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    H, W = frames.shape[1], frames.shape[2]

    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2])  # largest face
        x  = max(0, int(x  - (large_box_coef - 1.0) / 2.0 * fw))
        y  = max(0, int(y  - (large_box_coef - 1.0) / 2.0 * fh))
        fw = min(int(fw * large_box_coef), W - x)
        fh = min(int(fh * large_box_coef), H - y)
    else:
        x, y, fw, fh = 0, 0, W, H  # fallback: full frame

    C = frames.shape[3]
    resized = np.zeros((len(frames), out_h, out_w, C), dtype=np.float32)
    for i, frame in enumerate(frames):
        crop = frame[y : y + fh, x : x + fw]
        if crop.size == 0:
            crop = frame
        resized[i] = cv2.resize(crop.astype(np.float32), (out_w, out_h),
                                interpolation=cv2.INTER_AREA)
    return resized

In [ ]:
# Discover subjects and read ground truth heart rate

all_dirs = sorted([
    d for d in glob.glob(os.path.join(RAW_DATA_PATH, "*"))
    if os.path.isdir(d) and os.path.basename(d) != "videos"
])
print(f"Found {len(all_dirs)} subject folders\n")

subjects = []

for subj_dir in all_dirs:
    subj_id  = os.path.basename(subj_dir)
    subj_key = subj_id.replace("_", "")

    session_path = subj_dir

    video_pattern = os.path.join(RAW_DATA_PATH, "videos", f"{subj_id}.mkv")
    video_files = glob.glob(video_pattern)
    
    if not video_files:
        print(f"No video found for {subj_id}, skipping.")
        continue
    video_path = video_files[0]

    subjects.append({
        "subj_id":      subj_id,
        "subj_key":     subj_key,
        "video_path":   video_path,
        "session_path": session_path,
    })
    print(f"  {subj_id}  video={os.path.basename(video_path)}")

print(f"\nTotal subjects: {len(subjects)}")

In [ ]:
# Data preprocessing
# Face crop -> DiffNormalized (3ch) + Standardized (3ch) = 6ch
# Chunk into clips of CHUNK_LENGTH frames, save as .npy

# Clear any previous preprocessed data
if os.path.exists(PREPROCESSED_PATH):
    shutil.rmtree(PREPROCESSED_PATH)
os.makedirs(PREPROCESSED_PATH)
print(f"Cleared and recreated: {PREPROCESSED_PATH}\n")

all_input_files = []

for subj in subjects:
    subj_key     = subj["subj_key"]
    video_path   = subj["video_path"]
    session_path = subj["session_path"]

    print(f"=== Processing {subj_key} ===")

    # Read video
    frames = read_video_frames(video_path)
    T = frames.shape[0]
    print(f"  Video: {T} frames @ {VIDEO_FPS} fps")

    # Read and resample PPG
    ppg_signal = read_ppg_synced(session_path, T)
    print(f"  PPG green: min={ppg_signal.min():.0f}, max={ppg_signal.max():.0f}")

    # Face crop + resize to 72x72
    frames_cropped = crop_face_resize(frames, IMG_H, IMG_W)

    # Compute DiffNormalized (3ch) and Standardized (3ch)
    diff_data = diff_normalize_data(frames_cropped)   # (T, H, W, 3)
    std_data  = standardized_data(frames_cropped)     # (T, H, W, 3)

    # Concatenate along channel axis -> 6 channels
    data_6ch = np.concatenate([diff_data, std_data], axis=-1)  # (T, H, W, 6)

    # DiffNormalized label
    label = diff_normalize_label(ppg_signal)  # (T,)

    # Chunk into clips
    clip_num = T // CHUNK_LENGTH
    data_clips  = np.array([data_6ch[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH] for i in range(clip_num)])
    label_clips = np.array([label[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH]    for i in range(clip_num)])

    # Save per-subject
    subj_dir = os.path.join(PREPROCESSED_PATH, subj_key)
    os.makedirs(subj_dir)

    subj_files = []
    for chunk_idx in range(clip_num):
        input_path = os.path.join(subj_dir, f"{subj_key}_input{chunk_idx}.npy")
        label_path = os.path.join(subj_dir, f"{subj_key}_label{chunk_idx}.npy")

        np.save(input_path, data_clips[chunk_idx])   # (CHUNK_LENGTH, H, W, 6)
        np.save(label_path, label_clips[chunk_idx])  # (CHUNK_LENGTH,)
        subj_files.append(input_path)

    all_input_files.extend(subj_files)
    print(f"  {clip_num} clips -> {subj_dir}\n")

print(f"Total clips saved: {len(all_input_files)}")
print("\nFolder structure:")
for subj in subjects:
    d = os.path.join(PREPROCESSED_PATH, subj["subj_key"])
    n = len(glob.glob(os.path.join(d, "*_input*.npy")))
    print(f"  {subj['subj_key']}/  ({n} clips)")

In [ ]:
# PyTorch Dataset  (NDCHW format, returns (D, 6, H, W) per clip)

class GroupADataset(Dataset):

    def __init__(self, input_files):
        self.inputs = sorted(input_files)
        self.labels = [
            f.replace("input", "label")
            for f in self.inputs
        ]

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        data  = np.float32(np.load(self.inputs[index]))   # (D, H, W, 6)
        label = np.float32(np.load(self.labels[index]))   # (D,)

        # NDHWC -> NDCHW: (D, H, W, 6) -> (D, 6, H, W)
        data = np.transpose(data, (0, 3, 1, 2))

        fname      = os.path.basename(self.inputs[index])
        split_idx  = fname.index("_")
        subject_id = fname[:split_idx]                     # e.g. "S000"
        chunk_id   = fname[split_idx + 6:].split(".")[0]  # +6 skips "_input"

        return data, label, subject_id, chunk_id


dataset = GroupADataset(all_input_files)
loader  = DataLoader(dataset, batch_size=4, shuffle=False, num_workers=4)

print(f"Dataset : {len(dataset)} clips")
print(f"Loader  : {len(loader)} batches (batch_size=4)")

In [ ]:
# Post-processing functions

def detrend(signal_in, lambda_val=100):
    """Smoothness-priors detrending (Tarvainen et al.)."""
    T_len = len(signal_in)
    H_mat = np.eye(T_len)
    ones  = np.ones(T_len)
    D_mat = (np.diag(ones[:-2], -2)
             - 2 * np.diag(ones[:-1], -1)
             + np.diag(ones))
    D_mat = D_mat[2:, :]
    inv   = np.linalg.inv(H_mat + lambda_val ** 2 * D_mat.T @ D_mat)
    return (H_mat - inv) @ signal_in


def bandpass_filter(sig, fs, low, high, order=1):
    """Zero-phase Butterworth bandpass filter."""
    b, a = signal.butter(order, [low / fs * 2, high / fs * 2], btype="bandpass")
    return signal.filtfilt(b, a, sig.astype(np.float64))


def fft_peak_hz(sig, fs, low, high):
    """Return dominant frequency (Hz) in [low, high] Hz via FFT."""
    N = 1
    while N < len(sig):
        N *= 2
    freqs, pxx = periodogram(sig, fs=fs, nfft=N, detrend=False)
    mask = (freqs >= low) & (freqs <= high)
    if not mask.any():
        return 0.0
    return float(freqs[mask][np.argmax(pxx[mask])])


def calculate_snr(pred_ppg, hr_label_bpm, fs, low_pass=0.6, high_pass=3.3):
    """Signal-to-noise ratio at HR harmonics vs background noise (dB)."""
    N = 1
    while N < len(pred_ppg):
        N *= 2
    freqs, pxx = periodogram(pred_ppg, fs=fs, nfft=N, detrend=False)

    f1  = hr_label_bpm / 60.0
    f2  = 2 * f1
    dev = 6.0 / 60.0  # +-6 bpm tolerance

    sig_mask   = (((freqs >= f1 - dev) & (freqs <= f1 + dev))
                  | ((freqs >= f2 - dev) & (freqs <= f2 + dev)))
    noise_mask = ((freqs >= low_pass) & (freqs <= high_pass) & ~sig_mask)

    sig_power   = pxx[sig_mask].sum()
    noise_power = pxx[noise_mask].sum()
    if noise_power == 0:
        return float("inf")
    return float(10.0 * np.log10(sig_power / noise_power))


def _reform_from_dict(chunk_dict):
    """Concatenate chunks in sorted key order into a 1-D array."""
    return np.concatenate([chunk_dict[k] for k in sorted(chunk_dict.keys())])


def process_bvp(pred_chunks, label_chunks, fs=30, diff_flag=True):
    """Post-process predicted and label BVP chunk dicts into HR estimates.

    When diff_flag=True (DiffNormalized labels), signals are cumsum'd
    before detrending to recover the original BVP waveform.
    """
    pred  = _reform_from_dict(pred_chunks).astype(np.float64)
    label = _reform_from_dict(label_chunks).astype(np.float64)

    if diff_flag:
        pred  = detrend(np.cumsum(pred),  100)
        label = detrend(np.cumsum(label), 100)
    else:
        pred  = detrend(pred,  100)
        label = detrend(label, 100)

    pred_processed  = bandpass_filter(pred,  fs, low=0.6, high=3.3)
    label_processed = bandpass_filter(label, fs, low=0.6, high=3.3)

    hr_pred  = fft_peak_hz(pred_processed,  fs, 0.6, 3.3) * 60.0
    hr_label = fft_peak_hz(label_processed, fs, 0.6, 3.3) * 60.0
    snr_db   = calculate_snr(pred_processed, hr_label, fs)

    return hr_pred, hr_label, snr_db, pred_processed

In [ ]:
# Main inference loop
# For each model in MODELS: load -> infer -> per-subject results -> aggregate metrics
# -> export metrics.json + ppg_results.csv into results_groupA/{model_name}/

FS = VIDEO_FPS

# lookup: subj_key -> subject info dict
subjects_by_key = {s["subj_key"]: s for s in subjects}

for model_display_name, model_class_key, model_rel_path in MODELS:
    model_path = os.path.join(REPO_ROOT, model_rel_path)
    print(f"\n{'='*70}")
    print(f"Model: {model_display_name}  ({model_class_key})")
    print(f"Weights: {model_path}")
    print(f"{'='*70}\n")

    # ---- Instantiate model ----
    if model_class_key == "DeepPhys":
        model = DeepPhys(img_size=IMG_H)
        frame_depth = None  # no TSM alignment needed
    elif model_class_key == "Tscan":
        model = TSCAN(frame_depth=TSCAN_FRAME_DEPTH, img_size=IMG_H)
        frame_depth = TSCAN_FRAME_DEPTH
    else:
        raise ValueError(f"Unknown model_class_key: {model_class_key}")

    # ---- Load weights ----
    state_dict = torch.load(model_path, map_location=DEVICE)

    # strip 'module.' prefix if present (DataParallel artifact)
    if any(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k[len("module."):]: v for k, v in state_dict.items()}

    model.load_state_dict(state_dict)
    model = model.to(DEVICE)
    model.eval()

    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model loaded. Total parameters: {num_params:,}")

    # ---- Inference ----
    bvp_preds_dict  = {}  # subj_key -> {chunk_id: np.ndarray (CHUNK_LENGTH,)}
    bvp_labels_dict = {}

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Inference [{model_display_name}]"):
            data_t, labels_t, batch_subjects, batch_chunk_ids = batch

            N, D, C, H, W = data_t.shape  # (N, CHUNK_LENGTH, 6, 72, 72)

            # Flatten batch + temporal -> (N*D, 6, H, W)
            data_flat   = data_t.view(N * D, C, H, W).to(DEVICE)
            labels_flat = labels_t.view(N * D)  # (N*D,)

            # For TS-CAN: trim to multiple of frame_depth (TSM alignment)
            if frame_depth is not None:
                trim = (N * D) // frame_depth * frame_depth
            else:
                trim = N * D  # DeepPhys: no alignment needed

            data_flat   = data_flat[:trim]
            labels_flat = labels_flat[:trim]

            pred = model(data_flat)            # (trim, 1)
            pred_np  = pred.squeeze(-1).cpu().numpy()      # (trim,)
            label_np = labels_flat[:trim].numpy()           # (trim,)

            for i in range(N):
                start = i * CHUNK_LENGTH
                end   = start + CHUNK_LENGTH
                if end > trim:
                    break

                subj = batch_subjects[i]
                cid  = int(batch_chunk_ids[i])

                if subj not in bvp_preds_dict:
                    bvp_preds_dict[subj]  = {}
                    bvp_labels_dict[subj] = {}

                bvp_preds_dict[subj][cid]  = pred_np[start:end]
                bvp_labels_dict[subj][cid] = label_np[start:end]

    print(f"\nInference complete. Subjects: {sorted(bvp_preds_dict.keys())}")

    # ---- Per-subject results ----
    per_subject_results = []
    hr_preds_all  = []
    hr_labels_all = [] 
    snr_all       = []

    print(f"\n{'Subject':<10} {'HR_pred':>10} {'HR_label':>10} {'HR_err':>8} {'SNR':>7}")
    print("-" * 55)

    for subj_key in sorted(bvp_preds_dict.keys()):
        hr_pred, hr_label, _, pred_processed = process_bvp(
            bvp_preds_dict[subj_key], bvp_labels_dict[subj_key],
            fs=FS, diff_flag=True
        )

        snr_db     = calculate_snr(pred_processed, hr_label, FS)
        hr_err     = hr_pred - hr_label

        # map subj_key ("S000") back to original ID ("S_000")
        subj_id = subj_key[0] + "_" + subj_key[1:]

        per_subject_results.append({
            "name":                subj_id,
            "predicted_heartrate": hr_pred,
            "label_heartrate":     hr_label, 
            "heartrate_error":     hr_err,
            "snr_db":              snr_db,
        })

        hr_preds_all.append(hr_pred)
        hr_labels_all.append(hr_label)
        snr_all.append(snr_db)

        print(f"{subj_id:<10} {hr_pred:>10.3f} {hr_label:>10.3f} {hr_err:>8.3f} {snr_db:>7.2f}")

    hr_preds_all  = np.array(hr_preds_all)
    hr_labels_all = np.array(hr_labels_all)
    snr_all       = np.array(snr_all)

    # ---- Aggregate metrics ----
    n = len(hr_preds_all)
    assert n > 0, "No subjects to evaluate."

    err   = hr_preds_all - hr_labels_all
    abs_e = np.abs(err)
    sq_e  = err ** 2
    rel_e = abs_e / (np.abs(hr_labels_all) + 1e-9)

    mae       = float(np.mean(abs_e))
    mae_se    = float(np.std(abs_e) / np.sqrt(n))

    rmse      = float(np.sqrt(np.mean(sq_e)))
    rmse_se   = float(np.sqrt(np.std(sq_e) / np.sqrt(n)))

    mape      = float(np.mean(rel_e) * 100.0)
    mape_se   = float(np.std(rel_e) / np.sqrt(n) * 100.0)

    if n >= 2:
        pearson_r  = float(np.corrcoef(hr_preds_all, hr_labels_all)[0, 1])
        pearson_se = float(np.sqrt(max(0.0, (1 - pearson_r ** 2) / (n - 2))))
    else:
        pearson_r, pearson_se = float("nan"), float("nan")

    mean_snr    = float(np.mean(snr_all))
    mean_snr_se = float(np.std(snr_all) / np.sqrt(n))

    print(f"\n--- Aggregate Metrics [{model_display_name}] ---")
    print(f"MAE     : {mae:.4f} +/- {mae_se:.4f} bpm")
    print(f"RMSE    : {rmse:.4f} +/- {rmse_se:.4f} bpm")
    print(f"MAPE    : {mape:.4f} +/- {mape_se:.4f} %")
    print(f"Pearson : {pearson_r:.4f} +/- {pearson_se:.4f}")
    print(f"SNR     : {mean_snr:.4f} +/- {mean_snr_se:.4f} dB")

    # ---- Export to per-model subdirectory ----
    model_output_dir = os.path.join(OUTPUT_DIR, model_display_name)
    os.makedirs(model_output_dir, exist_ok=True)

    metrics_dict = {
        "model":      model_display_name,
        "n_subjects": n,
        "evaluation_method": "FFT BVP-derived HR",
        "bvp_bandpass_hz":   [0.6, 3.3],
        "aggregate_metrics": {
            "MAE":     {"value": mae,       "se": mae_se,      "unit": "bpm"},
            "RMSE":    {"value": rmse,      "se": rmse_se,     "unit": "bpm"},
            "MAPE":    {"value": mape,      "se": mape_se,     "unit": "%"},
            "Pearson": {"value": pearson_r, "se": pearson_se,  "unit": ""},
            "SNR":     {"value": mean_snr,  "se": mean_snr_se, "unit": "dB"},
        },
        "per_subject": [
            {
                "name":                r["name"],
                "predicted_heartrate": r["predicted_heartrate"],
                "label_heartrate":     r["label_heartrate"], 
                "heartrate_error":     r["heartrate_error"],
                "snr_db":              r["snr_db"],
            }
            for r in per_subject_results
        ],
    }

    json_path = os.path.join(model_output_dir, "metrics.json")
    with open(json_path, "w") as fh:
        json.dump(metrics_dict, fh, indent=2)
    print(f"\nMetrics saved to: {json_path}")

    # ---- Export ppg_results.csv ----
    csv_rows = []
    for r in per_subject_results:
        csv_rows.append({
            "name":                r["name"],
            "predicted_heartrate": r["predicted_heartrate"],
            "label_heartrate":     r["label_heartrate"], 
            "heartrate_error":     r["heartrate_error"]
        })

    results_df = pd.DataFrame(csv_rows, columns=[
        "name", "predicted_heartrate", "label_heartrate", "heartrate_error"
    ])

    csv_path = os.path.join(model_output_dir, "ppg_results.csv")
    results_df.to_csv(csv_path, index=False)

    print(f"CSV saved to: {csv_path}")
    print()
    print(results_df.to_string(index=False))

print(f"\n\nAll models processed. Results in: {OUTPUT_DIR}")

In [ ]:
# convert metric.json to csv

import os
import glob
import json
import pandas as pd

# 1. Khai báo đường dẫn gốc chứa các thư mục model dựa trên ảnh của bạn
ROOT_DIR = OUTPUT_DIR

# 2. Tìm tất cả các file metrics.json nằm trong các thư mục con
json_files = glob.glob(os.path.join(ROOT_DIR, "*", "metrics.json"))

data_rows = []

# 3. Lặp qua từng file JSON để lấy dữ liệu
for file_path in json_files:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        
        model_name = data.get("model", "Unknown")
        n_subjects = data.get("n_subjects", 0)
        metrics = data.get("aggregate_metrics", {})
        
        # Lấy giá trị MAE gốc để làm tiêu chí sắp xếp (Rank)
        mae_raw = metrics.get("MAE", {}).get("value", float('inf'))
        
        # Hàm định dạng chữ theo chuẩn "value +/- se" (làm tròn 2 chữ số)
        def format_metric(m):
            if not m: return ""
            return f"{m.get('value', 0):.2f} +/- {m.get('se', 0):.2f}"

        # Đẩy dữ liệu vào 1 hàng (row)
        row = {
            "model": model_name,
            "# subjects": n_subjects,
            "MAE_raw": mae_raw, # Cột tạm để sort
            "MAE (bpm)": format_metric(metrics.get("MAE")),
            "RMSE (bpm)": format_metric(metrics.get("RMSE")),
            "MAPE (%)": format_metric(metrics.get("MAPE")),
            "Pearson": format_metric(metrics.get("Pearson")),
            "SNR (dB)": format_metric(metrics.get("SNR")),
        }
        data_rows.append(row)

# 4. Chuyển thành DataFrame (bảng)
df = pd.DataFrame(data_rows)

if not df.empty:
    # Sắp xếp bảng theo giá trị MAE thô (từ thấp nhất -> cao nhất)
    df = df.sort_values(by="MAE_raw", ascending=True).reset_index(drop=True)
    
    # Thêm cột 'rank' vào vị trí đầu tiên (bắt đầu từ 1)
    df.insert(0, "rank", df.index + 1)
    
    # Xóa cột 'MAE_raw' vì không cần hiển thị ra CSV
    df = df.drop(columns=["MAE_raw"])
    
    # 5. Xuất ra file CSV
    out_csv_path = os.path.join(ROOT_DIR, "Model_Performance_Metrics.csv")
    df.to_csv(out_csv_path, index=False)
    
    print(f"✅ Đã gom thành công {len(json_files)} file JSON!")
    print(f"✅ File tổng hợp được lưu tại:\n{out_csv_path}\n")
    print("Preview dữ liệu:")
    print(df.head().to_string(index=False))
else:
    print("❌ Không tìm thấy file metrics.json nào trong thư mục!")